#Load Data

In [2]:
! pip install pandas 

  Using cached pandas-2.3.3-cp39-cp39-macosx_11_0_arm64.whl (10.8 MB)
  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_arm64.whl (5.3 MB)
  Using cached pytz-2026.2-py2.py3-none-any.whl (510 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl (349 kB)
You should consider upgrading via the '/Users/indrapal/industrial_predictive_alert_system/venv/bin/python3 -m pip install --upgrade pip' command.


In [7]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

df = pd.read_sql("SELECT * FROM sensor_data", engine)



In [6]:
df.head()

,id,temperature,pressure,vibration,humidity,voltage,current,failure,timestamp
0,1,91.30,13.82,4.60,85.21,222.04,13.61,0,2026-06-12 08:20:18.994006
1,2,49.23,37.32,0.71,61.83,226.24,12.53,0,2026-06-12 08:20:24.009272
2,3,99.19,16.87,6.62,49.21,236.64,17.11,1,2026-06-12 08:20:29.020066
3,4,29.98,18.00,2.08,80.77,243.20,8.66,0,2026-06-12 08:20:34.032468
4,5,31.21,31.42,5.85,31.61,210.67,2.27,0,2026-06-12 08:20:39.041930


In [10]:
df.shape

(9074, 9)

In [13]:
df.describe()

,id,temperature,pressure,vibration,humidity,voltage,current,failure,timestamp
count,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074.000000,9074
mean,4537.500000,60.324888,29.972908,5.011410,55.010737,230.002536,10.514447,0.078246,2026-06-12 12:50:00.648223744
min,1.000000,20.000000,10.000000,0.000000,20.000000,210.000000,1.000000,0.000000,2026-06-12 08:20:18.994006
25%,2269.250000,40.902500,20.090000,2.530000,37.610000,219.940000,5.670000,0.000000,2026-06-12 12:49:57.073638656
50%,4537.500000,60.630000,30.185000,5.040000,55.220000,230.040000,10.595000,0.000000,2026-06-12 12:50:52.486258944
75%,6805.750000,79.730000,39.827500,7.490000,72.565000,240.190000,15.210000,0.000000,2026-06-12 12:50:56.301371648
max,9074.000000,100.000000,50.000000,10.000000,90.000000,250.000000,20.000000,1.000000,2026-06-12 12:51:07.785801
std,2619.582505,22.757125,11.456672,2.885694,20.202764,11.604581,5.511422,0.268572,NaN


# process Data

In [14]:
df.isnull().sum()

id             0
temperature    0
pressure       0
vibration      0
humidity       0
voltage        0
current        0
failure        0
timestamp      0
dtype: int64

In [17]:
df = df.drop(columns=["timestamp", "id"])

In [15]:
df["risk_score"] = (
    (df["temperature"] > 85).astype(int)
    + (df["vibration"] > 6).astype(int)
    + (df["pressure"] > 40).astype(int)
)

In [19]:
df["temp_vibration"] = (
    df["temperature"] * df["vibration"]
)

In [21]:
df["power"] = (
    df["voltage"] * df["current"]
)

In [22]:
df["temp_pressure_ratio"] = (
    df["temperature"] / (df["pressure"] + 1)
)

In [23]:
df["high_temp"] = (df["temperature"] > 85).astype(int)

df["high_vibration"] = (
    df["vibration"] > 6
).astype(int)

df["high_pressure"] = (
    df["pressure"] > 40
).astype(int)

In [24]:
df.head()

,temperature,pressure,vibration,humidity,voltage,current,failure,risk_score,temp_vibration,power,temp_pressure_ratio,high_temp,high_vibration,high_pressure
0,91.30,13.82,4.60,85.21,222.04,13.61,0,1,419.9800,3021.9644,6.160594,1,0,0
1,49.23,37.32,0.71,61.83,226.24,12.53,0,0,34.9533,2834.7872,1.284708,0,0,0
2,99.19,16.87,6.62,49.21,236.64,17.11,1,2,656.6378,4048.9104,5.550644,1,1,0
3,29.98,18.00,2.08,80.77,243.20,8.66,0,0,62.3584,2106.1120,1.577895,0,0,0
4,31.21,31.42,5.85,31.61,210.67,2.27,0,0,182.5785,478.2209,0.962677,0,0,0


In [25]:
X = df[
    [
        "temperature",
        "pressure",
        "vibration",
        "humidity",
        "voltage",
        "current",
        "risk_score",
        "temp_vibration",
        "power",
        "temp_pressure_ratio"
    ]
]

y = df["failure"]

# model training

In [30]:
! pip install scikit-learn

  Using cached scikit_learn-1.6.1-cp39-cp39-macosx_12_0_arm64.whl (11.1 MB)
  Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Using cached scipy-1.13.1-cp39-cp39-macosx_12_0_arm64.whl (30.3 MB)
You should consider upgrading via the '/Users/indrapal/industrial_predictive_alert_system/venv/bin/python3 -m pip install --upgrade pip' command.


In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [32]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1673
           1       1.00      1.00      1.00       142

    accuracy                           1.00      1815
   macro avg       1.00      1.00      1.00      1815
weighted avg       1.00      1.00      1.00      1815


Confusion Matrix
[[1673    0]
 [   0  142]]


In [34]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
7,temp_vibration,0.298688
0,temperature,0.267547
6,risk_score,0.193946
2,vibration,0.149157
1,pressure,0.044606
9,temp_pressure_ratio,0.042375
3,humidity,0.001127
8,power,0.001061
5,current,0.000756
4,voltage,0.000737


In [35]:
import joblib

joblib.dump(model, "model.pkl")

['model.pkl']